# 01 Data Loading

This notebook contains **Steps 1A–1E** of the project workflow:

- **1A**: environment setup / library installation
- **1B**: optional Google Drive mount for Colab
- **1C**: project path setup and CSV verification
- **1D**: load CSV files into DuckDB
- **1E**: verify loaded tables

**GitHub note:** this notebook preserves the original workflow, but the Google Drive mount is only needed in Colab. For local or GitHub-based use, replace the project path with your local data directory.


# Install Required Libraries





In [1]:
# Run this only if the required packages are not already installed.
# In local environments, prefer installing from requirements.txt.
!pip install duckdb pandas pyarrow


* We have installed required libraries to create a database and store all the
dataset files for faster access.

# Step 1A: Mount Google Drive

In [2]:
# Optional: use this cell only when running in Google Colab.
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


We have mounted Google Drive to access the MIMIC-IV heart failure dataset files stored in the project folder.

# Step 1B: Set Data Path and Verify Files

In [3]:
DATA_PATH = "/content/drive/MyDrive/DataScience_Capstone_Project/"

import os
print(os.listdir(DATA_PATH))


['heart_diagnoses.csv', 'heart_diagnoses_all.csv', 'heart_diagnoses_all_true.csv', 'heart_procedures.csv', 'heart_microbiologyevents_first_micro.csv', 'heart_microbiologyevents.csv', 'heart_labevents_first_lab.csv', 'heart_labevents_examination_group.csv', 'README.md', 'LICENSE.txt', 'SHA256SUMS.txt', 'patients.csv', 'admissions.csv', 'diagnoses_icd.csv', 'RAG_data', 'hf_project.duckdb', 'hf_project.duckdb.wal']


* Defined the project data path and lists all available CSV files and database to confirm the dataset is accessible.

# Step 1C: Loaded All CSV Files into DuckDB Database

In [4]:
import duckdb
import os

DATA_PATH = "/content/drive/MyDrive/DataScience_Capstone_Project/"
DB_PATH = DATA_PATH + "hf_project.duckdb"

con = duckdb.connect(DB_PATH)

csv_files = [f for f in os.listdir(DATA_PATH) if f.endswith(".csv")]

for file in csv_files:
    table_name = file.replace(".csv", "")
    print(f"Loading {file} as table: {table_name}")

    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{DATA_PATH}{file}', IGNORE_ERRORS=true);
    """)

print("Database created at:", DB_PATH)


Loading heart_diagnoses.csv as table: heart_diagnoses
Loading heart_diagnoses_all.csv as table: heart_diagnoses_all
Loading heart_diagnoses_all_true.csv as table: heart_diagnoses_all_true
Loading heart_procedures.csv as table: heart_procedures
Loading heart_microbiologyevents_first_micro.csv as table: heart_microbiologyevents_first_micro
Loading heart_microbiologyevents.csv as table: heart_microbiologyevents
Loading heart_labevents_first_lab.csv as table: heart_labevents_first_lab
Loading heart_labevents_examination_group.csv as table: heart_labevents_examination_group
Loading patients.csv as table: patients
Loading admissions.csv as table: admissions


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loading diagnoses_icd.csv as table: diagnoses_icd


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Database created at: /content/drive/MyDrive/DataScience_Capstone_Project/hf_project.duckdb


* Connected to a DuckDB database on Google Drive and loads all 11
CSV files as tables using `read_csv_auto`. This creates a local analytical database for efficient SQL-based querying throughout the project.

# Step 1D: Verify Loaded Tables

In [5]:
con.execute("SHOW TABLES").fetchdf()


,name
0,admissions
1,diagnoses_icd
2,heart_diagnoses
3,heart_diagnoses_all
4,heart_diagnoses_all_true
5,heart_labevents_examination_group
6,heart_labevents_first_lab
7,heart_microbiologyevents
8,heart_microbiologyevents_first_micro
9,heart_procedures


Listed all tables currently stored in the DuckDB database to confirm successful data loading. The database contains 15 tables including raw data tables and derived feature tables from previous pipeline steps.

# Step 1E: Display Table Names

In [6]:

tables = con.execute("SHOW TABLES").fetchdf()
print(tables)

                                    name
0                             admissions
1                          diagnoses_icd
2                        heart_diagnoses
3                    heart_diagnoses_all
4               heart_diagnoses_all_true
5      heart_labevents_examination_group
6              heart_labevents_first_lab
7               heart_microbiologyevents
8   heart_microbiologyevents_first_micro
9                       heart_procedures
10                            hf_labeled
11                          lab_features
12                        model_features
13                        other_features
14                              patients


Printed the full list of table names in a readable format to verify the database schema before proceeding to cohort construction.